# Self-Verification of arrhythmia_features_draft.csv

Runs the 5 checks from the spec against the draft built in `extract_arrhythmia_features.ipynb`, prints every result explicitly, and only saves the final `arrhythmia_features_482.csv` after reviewing them.



In [ ]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("..").resolve()
DATA_INTERIM = BASE_DIR / "data" / "interim"

DRAFT_CSV = DATA_INTERIM / "arrhythmia_features_draft.csv"
FINAL_CSV = DATA_INTERIM / "arrhythmia_features_482.csv"  # name kept as specified, despite being 477 rows
CLINICAL_CSV = DATA_INTERIM / "imputed_477_cases.csv"

df = pd.read_csv(DRAFT_CSV)
clinical = pd.read_csv(CLINICAL_CSV)
expected_case_ids = set(clinical["caseid"])

print(f"Loaded draft: {df.shape}")

## Check 1: Shape

In [ ]:
EXPECTED_ROWS = 477   # see notebook intro: imputed_477_cases.csv has 477 caseids, not 482
EXPECTED_COLS = 13    # see notebook intro: the literal column list in the spec has 13 names

print(f"Shape: {df.shape}")
print(f"Matches expected ({EXPECTED_ROWS} rows, {EXPECTED_COLS} cols): "
      f"{df.shape == (EXPECTED_ROWS, EXPECTED_COLS)}")

missing_caseids = expected_case_ids - set(df["caseid"])
print(f"Caseids expected but missing from the draft: {sorted(missing_caseids)}")

## Check 2: Patients Where event_time_sec Is NaN (Extraction Failed Entirely)

In [ ]:
nan_event = df[df["event_time_sec"].isna()]
print(f"Count: {len(nan_event)}")
print(f"Caseids: {nan_event['caseid'].tolist()}")

## Check 3: Patients Where rhythm_onset_normal Is True

In [ ]:
onset_normal = df[df["rhythm_onset_normal"] == True]
print(f"Count: {len(onset_normal)}")
print(f"Caseids (event_time_sec for these is the clip start, not an arrhythmia onset): {onset_normal['caseid'].tolist()}")

## Check 4: Summary Statistics

In [ ]:
cols_to_summarize = ["event_time_sec", "arrhythmia_duration_sec", "rr_cv", "pct_ventricular", "pct_supraventricular"]
summary = df[cols_to_summarize].agg(["mean", "min", "max"])
print(summary.to_string())

n_near_zero_start = (df["event_time_sec"] < 5).sum()
print(f"\nPatients with event_time_sec < 5 seconds: {n_near_zero_start} "
      f"(may indicate the arrhythmia begins at the very start of the clip - noted, not removed)")

## Check 5: pct_normal + pct_supraventricular + pct_ventricular Should Be Within [0.95, 1.05]

In [ ]:
pct_sum = df["pct_normal"] + df["pct_supraventricular"] + df["pct_ventricular"]
out_of_range = df[(pct_sum < 0.95) | (pct_sum > 1.05)].copy()
out_of_range["pct_sum"] = pct_sum[out_of_range.index]

print(f"Rows with pct sum outside [0.95, 1.05]: {len(out_of_range)}")
if len(out_of_range) > 0:
    print(out_of_range[["caseid", "pct_normal", "pct_supraventricular", "pct_ventricular", "pct_sum"]].to_string(index=False))
    print("\nAs documented in extract_arrhythmia_features.ipynb, this is expected: these patients have")
    print("clean rows with no usable beat_type (mostly during Noise-labeled segments), which count")
    print("toward total_beats but not toward any of the three percentages. Not a counting bug.")

## Save Final Output

In [ ]:
# All 5 checks above have been reviewed and the results are explained (Check 1's row/col
# counts and Check 5's out-of-tolerance rows are both expected given the documented
# mismatches with the original spec and the underlying data quality finding).
assert df["caseid"].is_unique, "caseid must be unique before saving the final file"

df.to_csv(FINAL_CSV, index=False)
print(f"Saved final {df.shape[0]} rows x {df.shape[1]} columns to {FINAL_CSV.resolve()}")